# **Import Utilities**

In [ ]:
import time
from anthropic import Anthropic
import sys

sys.path.append('..')  # Add the parent directory of 'Stage_1-LLM_Ratings' to the Python path
from Utils.llm_evaluation_utils import *

api_key = os.environ.get('ANTHROPIC_API_KEY')
client = Anthropic(api_key=api_key)

MODEL = 'claude-sonnet-4-20250514'
MAX_TOKENS = 50
TEMPERATURE = 1
SYSTEM_MESSAGE = RATING_SYSTEM_MESSAGE

NUM_TRIALS = 1

In [ ]:
DATASET_DIR = Path("../../Dataset/cleaned_dataset")
SELECTED_TASKS = "1000_screens-few_shots"

# Input files
uicrit_data_file = DATASET_DIR / "uicrit_notna_deduped.parquet"
base64_screens_file = DATASET_DIR / "base64_screens.parquet"
few_shot_samples_dir = Path("../few_shot_samples")
few_shot_samples_file = few_shot_samples_dir / "few_shot_samples_df.parquet"

# Outputs
text_responses_jsonl_file = Path(f"./claude_4_0-responses-{SELECTED_TASKS}.jsonl")

all_results_dir     = Path(f"../Results/{SELECTED_TASKS}")
model_results_file  = Path(f"claude_4_0-ratings-{SELECTED_TASKS}.parquet")

# Load Data

In [ ]:
base64_screens_df = pd.read_parquet(base64_screens_file)

few_shot_samples_df = pd.read_parquet(few_shot_samples_file)

responses_df = load_responses_df(
    responses_dir=all_results_dir,
    results_file_name=model_results_file,
    uicrit_df=uicrit_data_file,
    num_trials=NUM_TRIALS
)
print('responses_df shape:', responses_df.shape)
responses_df.head()

responses_df shape: (997, 8)


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,7.0,4.0,4.0,8.0,8.0
1,28_T01,28,Enter details to Sing In to Scotiabank.,7.0,5.0,5.0,10.0,8.0
2,67_T01,67,Upload the Mississippi River Delta image to Ea...,8.0,1.0,1.0,2.0,8.0
3,190_T01,190,Click the 'Chat' button to initiate a conversa...,3.0,5.0,5.0,10.0,4.0
4,193_T01,193,Search and explore friends,3.0,4.0,4.0,8.0,6.0


In [4]:
# Get first screen_task_id per screen_id (including those with NaNs)
first_tasks = responses_df.drop_duplicates(subset='screen_id', keep='first')

# Get the corresponding screen_task_ids
screen_task_ids = first_tasks['screen_task_id'].unique()

# Filter the full responses_df to keep all rows matching those screen_task_ids
screen_responses_df = responses_df[responses_df['screen_task_id'].isin(screen_task_ids)].copy()
screen_responses_df = screen_responses_df.reset_index(drop=True)

del responses_df
responses_df = screen_responses_df
print('responses_df shape:', responses_df.shape)
responses_df.iloc[9:11]

responses_df shape: (1000, 8)


,screen_task_id,screen_id,task,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
9,445_T01,445,Expand the functionality by adding a new feat...,None,None,None,None,None
10,640_T01,640,Choose a station on World FM Radio.,None,None,None,None,None


## !!Temporary

In [6]:
# keep only the first screen_task_id of each screen_id
responses_df = responses_df.drop_duplicates(subset=['screen_id'], keep='first')
# keep first five rows for testing
responses_df = responses_df.head(5)
responses_df.reset_index(drop=True, inplace=True)
responses_df.head(5)

,screen_task_id,screen_id,task,trial,aesthetics_rating,learnability,efficiency,usability_rating,design_quality_rating
0,15_T01,15,Plan and Start Full Body Workouts,1,None,None,None,None,None
1,28_T01,28,Enter details to Sing In to Scotiabank.,1,None,None,None,None,None
2,67_T01,67,Upload the Mississippi River Delta image to Ea...,1,None,None,None,None,None
3,190_T01,190,Click the 'Chat' button to initiate a conversa...,1,None,None,None,None,None
4,193_T01,193,Search and explore friends,1,None,None,None,None,None


# Get Claude API Responses

In [ ]:
import base64

def is_base64_png(base64_string):
    """
    Returns True if the base64 string decodes to a PNG image, else False.
    """
    try:
        img_bytes = base64.b64decode(base64_string)
        # PNG signature: 89 50 4E 47 0D 0A 1A 0A
        return img_bytes.startswith(b'\x89PNG\r\n\x1a\n')
    except Exception as e:
        print("Error decoding base64:", e)
        return False

# Example usage:
print(is_base64_png(base64_screens_df.iloc[19]['base64_screen']))

True


In [ ]:
def save_json_response(response, screen_id, screen_task_id, trial, metric, jsonl_responses_file):
    # If `response` is a JSON string, parse it
    if isinstance(response, str):
        try:
            response = json.loads(response)
        except json.JSONDecodeError:
            print("Error decoding response JSON")
            return

    # Final record to save
    response_data = {
        'screen_id': screen_id,
        'screen_task_id': screen_task_id,
        'trial': trial,
        'metric': metric,
        **response  # Flatten the response into the top-level
    }

    # Save to JSONL
    with open(jsonl_responses_file, 'a') as f:
        f.write(json.dumps(response_data) + '\n')

def query_claude_model(
    client,
    model_name,
    prompt,
    current_base64,
    max_tokens=250,
    temperature=1,
    system="You are a UI expert",
    examples_df=None,
    assistant=None,
    image_media_type="png"
):
    """
    Sends a prompt and images to the specified large language model and returns its response.

    Args:
        client: Anthropic client instance.
        model_name: Model name string.
        prompt: Prompt text.
        current_base64: Base64 string of the current screen image.
        examples_df: Optional DataFrame with "base64_screen" column for few-shot samples.
        max_tokens: Max tokens for response.
        temperature: Sampling temperature.
        system: Optional system message.
        assistant: Optional assistant message.
        image_media_type: MIME type for images ("png" or "jpeg").
    """
    content_array = []

    # Add prompt text
    content_array.append({'type': 'text', 'text': prompt})

    # Add few-shot sample images if provided
    if examples_df is not None:
        for _, sample in examples_df.iterrows():
            content_array.append({
                'type': 'image',
                'source': {
                    'type': 'base64',
                    'media_type': "image/" + image_media_type,
                    'data': sample['base64_screen']
                }
            })

    # Add current screen image
    content_array.append({
        'type': 'image',
        'source': {
            'type': 'base64',
            'media_type': "image/" + image_media_type,
            'data': current_base64
        }
    })

    messages = [{'role': 'user', 'content': content_array}]
    if assistant is not None:
        messages.append({'role': 'assistant', 'content': assistant})

    response = client.messages.create(
        model=model_name,
        max_tokens=max_tokens,
        temperature=temperature,
        system=system,
        messages=messages
    )
    return response.content[0].text

In [ ]:
# Calculate the delay based on your rate limit
requests_limit_per_minute = 1000
delay = 60.0 / requests_limit_per_minute

incomplete_rows = responses_df[responses_df[EVALUATION_MAIN_ASPECTS].isnull().any(axis=1)]

for index, row in incomplete_rows.iterrows():
    screen_id = row['screen_id']
    # skip if screen_id is in column screen_id in few_shot_samples_df 
    if screen_id in few_shot_samples_df['screen_id'].values:
        # drop that row from responses_df
        responses_df = responses_df[responses_df['screen_id'] != screen_id]
        continue
    screen_task_id = row['screen_task_id']
    base64_string = base64_screens_df.loc[screen_id, 'base64_screen']

    print(f"Processing: index={index} | screen_task_id={screen_task_id}")

    for aspect in EVALUATION_MAIN_ASPECTS:
        if pd.notnull(row[aspect]):  # Skip if already filled
            continue

        prompt = build_rating_prompt(row['task'], GUIDELINES, evaluate=aspect, prompting_type='few-shot', samples=few_shot_samples_df)
        response = query_claude_model(client, MODEL, prompt, base64_string,
                                      MAX_TOKENS, TEMPERATURE, SYSTEM_MESSAGE,
                                      few_shot_samples_df, image_media_type="jpeg")

        save_json_response(response, screen_id, screen_task_id, trial=None, metric=aspect, jsonl_responses_file=text_responses_jsonl_file)
        update_rating_in_df(responses_df, screen_task_id, trial=None, metric=aspect, response=response)

        time.sleep(delay)

Processing: index=20 | screen_task_id=1369_T01


# Explore Results

In [ ]:
responses_df.head(10)

In [ ]:
# display_from_index = 0
# index_of_q1 = responses_df.columns.get_loc("Q1")

# responses_df.iloc[display_from_index:, index_of_q1:index_of_q1+15].head()

In [32]:
columns_with_none = (responses_df.isna() | (responses_df == '')).sum()
columns_with_none

screen_task_id           0
screen_id                0
task                     0
aesthetics_rating        0
learnability             0
efficiency               0
usability_rating         0
design_quality_rating    0
dtype: int64

In [ ]:
rows_with_none = responses_df[responses_df.isna().any(axis=1)]
rows_with_none

# Store Results

In [33]:
os.makedirs(all_results_dir, exist_ok=True) # Ensure the directory exists

parquet_output_file = os.path.join(all_results_dir, model_results_file)

responses_df.to_parquet(parquet_output_file, index=False)